<a href="https://colab.research.google.com/github/rubenoliva-dominguez/curso_cartagena/blob/main/dia4-1/4_decoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ejemplo CodeGen decoder

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# cargamos el tokenizador y el modelo de CodeGen, algo antiguo pero sirve
# para mostrar el por qué hay que usar SFT
tokenizer = AutoTokenizer.from_pretrained("Salesforce/codegen-350M-mono")
# lo cargamos en bfloat16 para ahorrar memoria
model = AutoModelForCausalLM.from_pretrained("Salesforce/codegen-350M-mono",
                                            trust_remote_code=True, revision="main",
                                            device_map="auto",
                                            dtype=torch.bfloat16)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/999 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/240 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/797M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/797M [00:00<?, ?B/s]

CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-mono
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Usar el modelo de manera naive

In [2]:
from transformers import set_seed
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
set_seed(42)

# esto no funciona, problemas de la forma de entender el modelo
PROMPT = """### Instruction:
{instruction}

### Response:
{response}"""

# generamos el input
text = PROMPT.format(
    instruction="compute the mean of a numpy array",
    response=""
)
# lo tokenizamos
tokenized = tokenizer(text, return_tensors="pt")
input_ids = tokenized.input_ids
attention_mask = tokenized.attention_mask
# generamos la continuación
generated_ids = model.generate(input_ids.cuda(), attention_mask=attention_mask.cuda(), max_length=128, do_sample=True, temperature=0.1,
                               eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id)
# mostramos el resultado
print()
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))



### Instruction:
compute the mean of a numpy array

### Response:

# 1.
#
# 2.
#
# 3.
#
# 4.
#
# 5.
#
# 6.
#
# 7.
#
# 8.
#
# 9.
#
# 10.
#
# 11.
#
# 12.
#
# 13.
#
# 14.
#
# 15.
#
# 16.
#
# 17.
#
# 18.
#



# Zero-shot prompting

In [3]:
# el truco es usar un prompt similar a cómo se entrenó el modelo
PROMPT = """
# {instruction}
{response}"""

text = PROMPT.format(
    instruction="compute the mean of a numpy array",
    response=""
)
# lo tokenizamos
tokenized = tokenizer(text, return_tensors="pt")
input_ids = tokenized.input_ids
attention_mask = tokenized.attention_mask
# generamos la continuación
generated_ids = model.generate(input_ids.cuda(), attention_mask=attention_mask.cuda(), max_length=128, do_sample=True, temperature=0.1,
                               eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id)
# mostramos el resultado
print()
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))



# compute the mean of a numpy array
def mean(arr):
    return np.mean(arr)

# compute the standard deviation of a numpy array
def std(arr):
    return np.std(arr)

# compute the median of a numpy array
def median(arr):
    return np.median(arr)

# compute the mode of a numpy array
def mode(arr):
    return np.mode(arr)

# compute the quartiles of a numpy array
def q1(arr):
    return np


# Few-shot prompting

In [4]:
PROMPT = """### Instruction:
sum two numbers in python

### Response:
a + b

### Instruction:
{instruction}

### Response:
{response}"""

text = PROMPT.format(
    instruction="compute the mean of a numpy array",
    response=""
)
# lo tokenizamos
tokenized = tokenizer(text, return_tensors="pt")
input_ids = tokenized.input_ids
attention_mask = tokenized.attention_mask
# generamos la continuación
generated_ids = model.generate(input_ids.cuda(), attention_mask=attention_mask.cuda(), max_length=128, do_sample=True, temperature=0.1,
                               eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.eos_token_id)
# mostramos el resultado
print()
print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))


### Instruction:
sum two numbers in python

### Response:
a + b

### Instruction:
compute the mean of a numpy array

### Response:
a.mean()

### Instruction:
compute the mean of a numpy array

### Response:
a.mean()

### Instruction:
compute the mean of a numpy array

### Response:
a.mean()

### Instruction:
compute the mean of a numpy array

### Response:
a.mean()

### Instruction:
compute the mean
